# Exploratory Data Analysis (EDA) 

**Dataset:** Water Potability (Kaggle)  
**Goal:** Understand the data structure, distributions, missing values, correlations, and class balance before modelling.


## 1. Imports & Setup

In [ ]:
import pandas as pd #data processing
import numpy as np # linear algebra 
import matplotlib.pyplot as plt
import seaborn as sns #visualization 
import plotly.express as px #visulaization
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/water_potability.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Basic Info

In [ ]:
df.info()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues')

## 4. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().reset_index()
missing.columns = ['Feature', 'Missing Count']
missing['Missing %'] = (missing['Missing Count'] / len(df) * 100).round(2)
missing = missing[missing['Missing Count'] > 0]
print(missing)

# Visualize
plt.figure(figsize=(8, 4))
sns.barplot(data=missing, x='Feature', y='Missing %', palette='Reds_r')
plt.title('Missing Values by Feature (%)')
plt.ylabel('Missing %')
plt.tight_layout()
plt.show()

## 5. Target Class Balance (Potability)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['Potability'].value_counts()
labels = ['Not Potable (0)', 'Potable (1)']
colors = ['#e74c3c', '#2ecc71']

axes[0].pie(counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Potability Distribution')

sns.countplot(data=df, x='Potability', palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title('Class Count')
axes[1].set_xticklabels(labels)

plt.tight_layout()
plt.show()

print(df['Potability'].value_counts())

## 6. Feature Distributions

In [ ]:
features = df.columns.drop('Potability')

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    for potability, color, label in zip([0, 1], ['#e74c3c', '#2ecc71'], ['Not Potable', 'Potable']):
        subset = df[df['Potability'] == potability][col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.6, color=color, label=label)
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.suptitle('Feature Distributions by Potability', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Boxplots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(data=df, x='Potability', y=col,
                palette=['#e74c3c', '#2ecc71'], ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xticklabels(['Not Potable', 'Potable'])

plt.suptitle('Boxplots by Potability (Outlier Detection)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 9. WHO Threshold Comparison

In [ ]:
who_limits = {
    'ph': (6.5, 8.5),
    'Sulfate': (0, 250),
    'Chloramines': (0, 4),
    'Turbidity': (0, 5),
    'Conductivity': (0, 400),
}

print('=' * 55)
print(f'{"Feature":<15} {"WHO Min":<10} {"WHO Max":<10} {"Data Mean":<12} {"Status"}')
print('=' * 55)

for feature, (low, high) in who_limits.items():
    col_name = feature if feature in df.columns else feature.capitalize()
    mean_val = df[col_name].mean()
    status = '✅ OK' if low <= mean_val <= high else '⚠️  OUT OF RANGE'
    print(f'{col_name:<15} {low:<10} {high:<10} {mean_val:<12.2f} {status}')

print('=' * 55)

## 10. Summary & Key Findings

In [ ]:
print('📋 EDA Summary')
print('=' * 50)
print(f'  Total samples       : {len(df)}')
print(f'  Features            : {df.shape[1] - 1}')
print(f'  Potable samples     : {(df["Potability"]==1).sum()} ({(df["Potability"]==1).mean()*100:.1f}%)')
print(f'  Not Potable samples : {(df["Potability"]==0).sum()} ({(df["Potability"]==0).mean()*100:.1f}%)')
print(f'  Features w/ NaN     : {df.isnull().any().sum()}')
print(f'  Total missing vals  : {df.isnull().sum().sum()}')
print('=' * 50)
print('Next step: Preprocessing (imputation + scaling)')